# Hybrid Pipeline: GEE (1980–2020) + Precomputed Local Rasters (2021–2025)

Produces one combined `temp_monthly_1980_2025_TOP3_WATERSHEDS_hybrid.csv` and
the cross-watershed yearly average, for **Navajo, Blue Mesa, Pueblo** reservoirs/basins.

- **1980–2020**: pulled live from Earth Engine's `OREGONSTATE/PRISM/AN81m` monthly collection (needs EE auth) — this is the only part that actually needs to run here.
- **2021–2025**: already computed from the 15 local `prism_tmean_us_25m_YYYYMM.tif` rasters
  (Apr/May/Jun, 2021–2025) and embedded directly below — no rasterio/geopandas raster
  masking, no re-uploading tifs.

**You only need the shapefile in Drive** (the 5 `co_top3_watersheds_combined.*` files),
used to build the watershed FeatureCollection for the Earth Engine query. No `rasterio`
needed anywhere in this version.

In [11]:
# --- 1. Setup: install + imports + mount Drive ---
!pip -q install geopandas earthengine-api

import ee
import io, glob
import numpy as np
import pandas as pd
import geopandas as gpd
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# --- 2. Authenticate Earth Engine ---
ee.Authenticate()
ee.Initialize(project='esiil-2026-500417')
print('Earth Engine initialized with project esiil-2026-500417')

Earth Engine initialized with project esiil-2026-500417


In [13]:
# --- 3. Load the top-3 watershed shapefile (needed for the GEE query) ---
hits = (glob.glob('/content/drive/MyDrive/**/co_top3_watersheds_combined.shp', recursive=True)
        + glob.glob('/content/co_top3_watersheds_combined.shp'))
assert hits, 'co_top3_watersheds_combined.shp not found — upload all 5 files (.shp .shx .dbf .prj .cpg) to Drive'
SHP = hits[0]
print('Using:', SHP)

watersheds = gpd.read_file(SHP).to_crs(4326)
print(f'{len(watersheds)} watershed(s):', sorted(watersheds['name'].tolist()))

features = [ee.Feature(ee.Geometry(row.geometry.__geo_interface__), {'name': row['name']})
            for _, row in watersheds.iterrows()]
watersheds_fc = ee.FeatureCollection(features)

Using: /content/drive/MyDrive/co_top3_watersheds_combined_shp/co_top3_watersheds_combined.shp
3 watershed(s): ['Blue Mesa Reservoir', 'Navajo Reservoir', 'Pueblo Reservoir']


In [14]:
# --- 4. GEE: basin-mean tmean, 1980-2020, PRISM monthly ---
# AN81m is deprecated and frozen at 2020-12, but 1980-2020 is exactly the span
# it covers, so it works fine here. Swap in 'OREGONSTATE/PRISM/ANm' if you
# prefer the current, maintained asset.
COLLECTION = 'OREGONSTATE/PRISM/AN81m'
YEARS_GEE = range(1980, 2021)
MONTHS = [4, 5, 6]
col = ee.ImageCollection(COLLECTION)

gee_rows = []
for y in YEARS_GEE:
    for m in MONTHS:
        start = ee.Date.fromYMD(y, m, 1)
        month_col = col.filterDate(start, start.advance(1, 'month')).select('tmean')
        if month_col.size().getInfo() == 0:
            print(f'{y}-{m:02d}: MISSING in GEE, skipping')
            for name in watersheds['name']:
                gee_rows.append({'watershed': name, 'year': y, 'month': m, 'tmean_c': None})
            continue
        stats = month_col.first().reduceRegions(collection=watersheds_fc,
                                                reducer=ee.Reducer.mean(),
                                                scale=4000).getInfo()
        for feat in stats['features']:
            p = feat['properties']
            gee_rows.append({'watershed': p['name'], 'year': y, 'month': m,
                             'tmean_c': p.get('mean')})
    print(f'{y}: done')

gee_df = pd.DataFrame(gee_rows)
gee_df.to_csv('/content/drive/MyDrive/prism_gee_monthly_1980_2020_TOP3.csv', index=False)
print('\nSaved prism_gee_monthly_1980_2020_TOP3.csv —', len(gee_df), 'rows')

1980: done
1981: done
1982: done
1983: done
1984: done
1985: done
1986: done
1987: done
1988: done
1989: done
1990: done
1991: done
1992: done
1993: done
1994: done
1995: done
1996: done
1997: done
1998: done
1999: done
2000: done
2001: done
2002: done
2003: done
2004: done
2005: done
2006: done
2007: done
2008: done
2009: done
2010: done
2011: done
2012: done
2013: done
2014: done
2015: done
2016: done
2017: done
2018: done
2019: done
2020: done

Saved prism_gee_monthly_1980_2020_TOP3.csv — 369 rows


## 2021–2025: precomputed local-raster results

These values were computed from your 15 uploaded `prism_tmean_us_25m_YYYYMM.tif`
files (Apr/May/Jun, 2021–2025) by masking each raster to each watershed polygon
and averaging. Embedded as CSV text below — this cell just loads it, no raster
libraries or file uploads needed.

In [15]:
# --- 5. LOCAL: precomputed basin-mean tmean, 2021-2025 ---
local_csv_text = """watershed,year,month,tmean_c
Navajo Reservoir,2021,4,5.927696780063562
Blue Mesa Reservoir,2021,4,1.7135761043864115
Pueblo Reservoir,2021,4,3.4699099151181505
Navajo Reservoir,2021,5,10.711187663497613
Blue Mesa Reservoir,2021,5,6.350056502501486
Pueblo Reservoir,2021,5,8.789528354005494
Navajo Reservoir,2021,6,17.06838833042449
Blue Mesa Reservoir,2021,6,13.183940049431197
Pueblo Reservoir,2021,6,15.71393042953921
Navajo Reservoir,2022,4,5.594538103857655
Blue Mesa Reservoir,2022,4,1.1728152777105347
Pueblo Reservoir,2022,4,3.977773691147146
Navajo Reservoir,2022,5,10.867660242599456
Blue Mesa Reservoir,2022,5,6.400829662622562
Pueblo Reservoir,2022,5,9.013443716835807
Navajo Reservoir,2022,6,16.056368742983764
Blue Mesa Reservoir,2022,6,12.04525696164808
Pueblo Reservoir,2022,6,14.732679048054655
Navajo Reservoir,2023,4,3.4437751712730815
Blue Mesa Reservoir,2023,4,-1.299143512387275
Pueblo Reservoir,2023,4,2.634860079976874
Navajo Reservoir,2023,5,10.678602817599758
Blue Mesa Reservoir,2023,5,6.698773776679105
Pueblo Reservoir,2023,5,9.404529197073318
Navajo Reservoir,2023,6,13.404527242929657
Blue Mesa Reservoir,2023,6,9.596115474435491
Pueblo Reservoir,2023,6,12.443694873930703
Navajo Reservoir,2024,4,5.525153643758263
Blue Mesa Reservoir,2024,4,1.581426995886012
Pueblo Reservoir,2024,4,4.709517074638689
Navajo Reservoir,2024,5,9.064642903263584
Blue Mesa Reservoir,2024,5,3.93889274037856
Pueblo Reservoir,2024,5,7.354722716774739
Navajo Reservoir,2024,6,17.23997506919814
Blue Mesa Reservoir,2024,6,13.07760462372251
Pueblo Reservoir,2024,6,16.45103611543145
Navajo Reservoir,2025,4,5.314568798478639
Blue Mesa Reservoir,2025,4,0.7637168818392289
Pueblo Reservoir,2025,4,4.050692842711865
Navajo Reservoir,2025,5,9.331616131562154
Blue Mesa Reservoir,2025,5,5.407209305118614
Pueblo Reservoir,2025,5,8.309648023356853
Navajo Reservoir,2025,6,16.223869708905678
Blue Mesa Reservoir,2025,6,12.322626229545943
Pueblo Reservoir,2025,6,15.292541274890093
"""

local_df = pd.read_csv(io.StringIO(local_csv_text))
local_df.to_csv('/content/drive/MyDrive/prism_local_monthly_2021_2025_TOP3.csv', index=False)
print('Loaded local_df —', len(local_df), 'rows (all 15 months present, no gaps)')
local_df.head()

Loaded local_df — 45 rows (all 15 months present, no gaps)


,watershed,year,month,tmean_c
0,Navajo Reservoir,2021,4,5.927697
1,Blue Mesa Reservoir,2021,4,1.713576
2,Pueblo Reservoir,2021,4,3.469910
3,Navajo Reservoir,2021,5,10.711188
4,Blue Mesa Reservoir,2021,5,6.350057


In [16]:
# --- 6. STITCH: combine GEE (1980-2020) + local (2021-2025) ---
monthly = (pd.concat([gee_df, local_df], ignore_index=True)
             .drop_duplicates(['watershed', 'year', 'month'], keep='last')  # local wins on overlap
             .sort_values(['watershed', 'year', 'month'])
             .reset_index(drop=True))

wide = (monthly.pivot(index=['watershed', 'year'], columns='month', values='tmean_c')
               .rename(columns={4: 'april_c', 5: 'may_c', 6: 'june_c'})
               .reset_index())

mcols = ['april_c', 'may_c', 'june_c']
complete = wide[mcols].notna().sum(axis=1)
wide['temp_aprjun_c'] = wide[mcols].mean(axis=1).where(complete == 3)

wide.to_csv('/content/drive/MyDrive/temp_monthly_1980_2025_TOP3_WATERSHEDS_hybrid.csv', index=False)
print(f'rows: {len(wide)} | watersheds: {wide.watershed.nunique()} | '
      f'years: {wide.year.min()}–{wide.year.max()}')
print(f'missing month-cells: {int(wide[mcols].isna().sum().sum())}')

rows: 138 | watersheds: 3 | years: 1980–2025
missing month-cells: 0


In [17]:
# --- 7. Cross-watershed average: one row per year ---
avg = (wide.groupby('year')[mcols].mean()
            .round(4)
            .rename(columns={'april_c': 'April', 'may_c': 'May', 'june_c': 'June'})
            .reset_index())

avg.to_csv('/content/drive/MyDrive/temp_3watersheds_mean_by_year_1980_2025_hybrid.csv', index=False)
print(avg.to_string(index=False))

 year   April     May    June
 1980  0.8275  6.2293 13.3995
 1981  5.3181  7.2155 14.4637
 1982  1.7849  6.5572 11.1962
 1983 -0.5959  5.4050 10.8155
 1984  0.1558  9.0738 11.8129
 1985  3.7638  8.0004 13.1571
 1986  3.1637  7.1167 12.9843
 1987  3.6294  7.5087 13.2343
 1988  3.2237  7.3885 13.8165
 1989  4.8606  8.4967 11.8845
 1990  4.0126  6.4976 14.3231
 1991  1.6261  7.5215 12.2847
 1992  5.3934  8.8036 11.4675
 1993  2.0032  7.5693 11.6691
 1994  2.8620  8.4857 14.9507
 1995  1.1317  5.3882 10.9293
 1996  2.9277 10.0872 13.6520
 1997  0.8269  7.7534 12.7363
 1998  1.0420  8.3936 11.7276
 1999  1.1866  6.4800 11.9610
 2000  4.9155  9.8265 14.0123
 2001  4.1522  9.1895 13.9564
 2002  5.5728  8.6547 15.8509
 2003  3.5223  8.6453 12.4282
 2004  3.7714  8.8101 13.0455
 2005  3.1851  8.6141 12.3951
 2006  4.8766  9.7925 14.9624
 2007  3.4527  8.2560 13.7816
 2008  1.7412  7.0611 13.1723
 2009  2.3261  9.8223 11.8197
 2010  3.3179  6.8607 14.5008
 2011  3.0900  5.9939 13.7426
 2012  5.6

In [19]:
# Display the 'wide' DataFrame to see individual watershed temperatures per month and year
display(wide)

month,watershed,year,april_c,may_c,june_c,temp_aprjun_c
0,Blue Mesa Reservoir,1980,-1.727521,4.205281,11.024199,4.500653
1,Blue Mesa Reservoir,1981,2.861120,5.068033,12.159920,6.696358
2,Blue Mesa Reservoir,1982,-0.817514,4.429652,9.247852,4.286663
3,Blue Mesa Reservoir,1983,-2.792399,3.285109,8.895138,3.129283
4,Blue Mesa Reservoir,1984,-2.642231,6.680009,9.270396,4.436058
...,...,...,...,...,...,...
133,Pueblo Reservoir,2021,3.469910,8.789528,15.713930,9.324456
134,Pueblo Reservoir,2022,3.977774,9.013444,14.732679,9.241299
135,Pueblo Reservoir,2023,2.634860,9.404529,12.443695,8.161028
136,Pueblo Reservoir,2024,4.709517,7.354723,16.451036,9.505092


In [21]:
# Save the 'wide' DataFrame to a CSV file in Google Drive
wide_file_path = '/content/drive/MyDrive/individual_watershed_temperatures.csv'
wide.to_csv(wide_file_path, index=False)
print(f"'wide' DataFrame saved to: {wide_file_path}")

'wide' DataFrame saved to: /content/drive/MyDrive/individual_watershed_temperatures.csv


In [22]:
from google.colab import files

file_to_download = '/content/drive/MyDrive/individual_watershed_temperatures.csv'
files.download(file_to_download)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>